# Parametric Testing — Reusable Project Template

Copy this notebook as the starting point for a new hypothesis-testing project. It defines a small toolkit of helper functions once, then walks through a standard pipeline: **load → explore → check assumptions → choose & run test → correct for multiple comparisons if needed → effect size / power → interpret**.

Fill in the `# >>> EDIT <<<` spots with your project's specifics; the helper functions below should work unchanged for most projects.


## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.weightstats import ttest_ind as sm_ttest_ind
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.power import TTestPower, TTestIndPower

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 4)


## 1. Toolkit — reusable helper functions
(Define once. Reuse across projects.)

In [ ]:
def check_normality(data, label=""):
    """Visual + formal normality checks. Returns a dict of test results."""
    data = np.asarray(data, dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    axes[0].hist(data, bins=min(20, max(5, len(data) // 5)), edgecolor="k")
    axes[0].set_title(f"Histogram {label}")
    stats.probplot(data, dist="norm", plot=axes[1])
    axes[1].set_title(f"QQ plot {label}")
    plt.tight_layout()
    plt.show()

    scaled = (data - data.mean()) / data.std()
    ks_stat, ks_p = stats.kstest(scaled, stats.norm.cdf)
    ad_result = stats.anderson(data, dist="norm")
    shapiro_stat, shapiro_p = stats.shapiro(data) if len(data) <= 5000 else (np.nan, np.nan)

    print(f"Kolmogorov-Smirnov: stat={ks_stat:.4f}, p={ks_p:.4f}")
    print(f"Anderson-Darling:   stat={ad_result.statistic:.4f} "
          f"(5% critical value={ad_result.critical_values[2]:.4f})")
    print(f"Shapiro-Wilk:       stat={shapiro_stat:.4f}, p={shapiro_p:.4f}  "
          f"{'(best for n < ~50)' if len(data) < 50 else '(use with caution for large n)'}")

    return {"ks_p": ks_p, "anderson": ad_result, "shapiro_p": shapiro_p}


def check_independence_sequence(data):
    """Durbin-Watson check -- only meaningful if row order is a genuine sequence (time, trial, etc.)."""
    dw = durbin_watson(np.asarray(data, dtype=float))
    print(f"Durbin-Watson = {dw:.4f}  (~2 = independent, ~0 = positive autocorrelation, ~4 = negative)")
    return dw


def check_equal_variance(*groups):
    """Levene's test across 2+ groups."""
    f_stat, p_value = stats.levene(*groups)
    print(f"Levene's test: F={f_stat:.4f}, p={p_value:.4f}  "
          f"-> {'variances look EQUAL' if p_value > 0.05 else 'variances look UNEQUAL'}")
    return f_stat, p_value


def f_test(a, b):
    """Fisher's F-test for equal variance between exactly two groups."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if np.var(a, ddof=1) > np.var(b, ddof=1):
        g1, g2 = a, b
    else:
        g1, g2 = b, a
    f_stat = np.var(g1, ddof=1) / np.var(g2, ddof=1)
    df1, df2 = g1.size - 1, g2.size - 1
    p_value = 2 * min(stats.f.cdf(f_stat, df1, df2), 1 - stats.f.cdf(f_stat, df1, df2))
    return f_stat, p_value, df1, df2


def cohens_d_two_sample(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    n1, n2 = len(a), len(b)
    pooled_sd = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / pooled_sd


def run_two_sample_test(a, b, alpha=0.05, alternative="two-sided"):
    """Checks variance equality, then runs the pooled or Welch t-test automatically."""
    f_stat, p_var, df1, df2 = f_test(a, b)
    equal_var = p_var > alpha
    print(f"F-test for equal variance: F={f_stat:.4f}, p={p_var:.4f} -> "
          f"using {'pooled' if equal_var else "Welch's"} t-test")
    t_stat, p_value = stats.ttest_ind(a, b, equal_var=equal_var, alternative=alternative)
    d = cohens_d_two_sample(a, b)
    print(f"t={t_stat:.4f}, p={p_value:.4g}, Cohen's d={d:.3f}")
    return {"t_stat": t_stat, "p_value": p_value, "cohens_d": d, "equal_var_used": equal_var}


def run_anova_with_posthoc(data, value_col, group_col, alpha=0.05):
    """Levene's test -> equal/unequal-variance ANOVA -> Tukey HSD if significant."""
    groups = [g[value_col].values for _, g in data.groupby(group_col)]
    f_stat, p_var = check_equal_variance(*groups)
    use_var = "equal" if p_var > alpha else "unequal"
    anova_result = anova_oneway(data[value_col], data[group_col], use_var=use_var)
    print(f"\n{use_var.capitalize()}-variance ANOVA: statistic={anova_result.statistic:.4f}, "
          f"p-value={anova_result.pvalue:.4g}")
    tukey = None
    if anova_result.pvalue <= alpha:
        tukey = pairwise_tukeyhsd(endog=data[value_col], groups=data[group_col], alpha=alpha)
        print("\nSignificant -- post-hoc Tukey HSD:")
        print(tukey)
    else:
        print("\nNot significant -- no post-hoc test needed.")
    return anova_result, tukey


def bonferroni_pairwise(groups_dict, alpha=0.05):
    """Runs all pairwise t-tests among a dict of {label: array} and Bonferroni-corrects the p-values."""
    from itertools import combinations
    labels = list(groups_dict.keys())
    rows = []
    for a, b in combinations(labels, 2):
        t_stat, p_value = stats.ttest_ind(groups_dict[a], groups_dict[b])
        rows.append({"comparison": f"{a} vs {b}", "t_stat": t_stat, "raw_p": p_value})
    df = pd.DataFrame(rows)
    reject, corrected_p, *_ = multipletests(df["raw_p"], alpha=alpha, method="bonferroni")
    df["bonferroni_p"] = corrected_p
    df["reject_H0"] = reject
    return df


## 2. Load your data

In [ ]:
# >>> EDIT <<<
# data = pd.read_csv("your_file.csv")
# data.head()


## 3. Explore
Descriptive stats, group means, and a quick plot before any testing.

In [ ]:
# >>> EDIT <<<
# data.describe()
# data.groupby("your_group_col")["your_value_col"].agg(["mean", "std", "count"])
# data.boxplot(column="your_value_col", by="your_group_col")


## 4. Check assumptions
Use the toolkit functions on each group you plan to compare.

In [ ]:
# >>> EDIT <<<
# for name, group in data.groupby("your_group_col"):
#     print(f"--- {name} ---")
#     check_normality(group["your_value_col"], label=str(name))

# check_equal_variance(*[g["your_value_col"].values for _, g in data.groupby("your_group_col")])


## 5. Choose & run the appropriate test

- **2 groups:** `run_two_sample_test(group_a, group_b)` (auto-selects pooled vs. Welch's)
- **3+ groups:** `run_anova_with_posthoc(data, value_col, group_col)` (auto-selects equal vs. Welch's ANOVA, runs Tukey if significant)
- **One sample vs. a fixed value:** `stats.ttest_1samp(data, popmean=..., alternative=...)`
- **Paired measurements:** `stats.ttest_rel(post, pre, alternative=...)`
- **Linear association between two continuous variables:** `stats.pearsonr(x, y)`


In [ ]:
# >>> EDIT <<< -- pick ONE of the patterns below and delete the rest

# Two independent groups:
# result = run_two_sample_test(group_a, group_b)

# 3+ groups:
# anova_result, tukey = run_anova_with_posthoc(data, value_col="your_value_col", group_col="your_group_col")

# One-sample:
# t_stat, p_value = stats.ttest_1samp(data["your_value_col"], popmean=SOME_VALUE, alternative="two-sided")

# Paired:
# t_stat, p_value = stats.ttest_rel(post_values, pre_values, alternative="two-sided")

# Correlation:
# r, p_value = stats.pearsonr(data["x_col"], data["y_col"])


## 6. Multiple comparisons (only if you ran several t-tests instead of ANOVA)

In [ ]:
# >>> EDIT <<<
# groups_dict = {name: g["your_value_col"].values for name, g in data.groupby("your_group_col")}
# bonferroni_pairwise(groups_dict, alpha=0.05)


## 7. Effect size & power
Always report effect size alongside p-value; consider whether your sample size gives adequate power.

In [ ]:
# >>> EDIT <<<
# For a two-sample comparison, cohens_d_two_sample(group_a, group_b) is already computed inside
# run_two_sample_test(). To check power retroactively or plan a follow-up study:

# power_calc = TTestIndPower()
# power_calc.solve_power(effect_size=observed_d, nobs1=len(group_a), alpha=0.05, ratio=len(group_b)/len(group_a))
# power_calc.solve_power(effect_size=observed_d, power=0.80, alpha=0.05, ratio=1.0)  # required n per group


## 8. Interpret & report

Answer in plain language:
1. What was the null hypothesis, and do you reject or fail to reject it?
2. What is the effect size, and is it practically meaningful (not just statistically significant)?
3. Were any assumptions borderline or violated? How does that affect your confidence in the result?
4. If you ran multiple tests, did you correct for that?
5. What would you tell a non-technical stakeholder in two sentences?


*(write your interpretation here)*